In [ ]:
import shutil
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
if not (NOTEBOOK_DIR / "organize.ipynb").exists():
    NOTEBOOK_DIR = Path("data_pipeline/scenario_assembly").resolve()
PIPELINE_DIR = NOTEBOOK_DIR.parent

# Source directories
ARTIFACTS_SRC = PIPELINE_DIR / "artifact_generation" / "all_artifacts_gemini-2.5-pro"
EVENTS_SRC = PIPELINE_DIR / "artifact_generation" / "all_events_gemini-2.5-pro"

# Target directory
TARGET_DIR = NOTEBOOK_DIR

# Domains to process
DOMAINS = ["research", "tutoring"]

def organize_files():
    """Organize events and artifacts files into the new structure."""

    stats = {
        'domains_processed': 0,
        'subdirs_processed': 0,
        'events_files_copied': 0,
        'artifacts_files_copied': 0,
        'errors': []
    }

    for domain in DOMAINS:
        print("
" + "=" * 80)
        print(f"Processing {domain.upper()} domain")
        print("=" * 80)

        artifacts_domain_path = ARTIFACTS_SRC / domain
        events_domain_path = EVENTS_SRC / domain
        target_domain_path = TARGET_DIR / domain

        if not artifacts_domain_path.exists():
            print(f"Warning: {artifacts_domain_path} does not exist, skipping...")
            continue
        if not events_domain_path.exists():
            print(f"Warning: {events_domain_path} does not exist, skipping...")
            continue

        target_domain_path.mkdir(parents=True, exist_ok=True)

        artifacts_subdirs = {d.name: d for d in artifacts_domain_path.iterdir() if d.is_dir()}
        events_subdirs = {d.name: d for d in events_domain_path.iterdir() if d.is_dir()}
        all_subdirs = set(artifacts_subdirs.keys()) | set(events_subdirs.keys())

        print(f"Found {len(all_subdirs)} subdirectories")
        print("Processing subdirectories...")

        for idx, subdir_name in enumerate(sorted(all_subdirs, key=lambda x: int(x) if x.isdigit() else float('inf')), 1):
            if idx % 20 == 0 or idx == len(all_subdirs):
                print(f"  Progress: {idx}/{len(all_subdirs)} subdirectories")

            target_subdir = target_domain_path / subdir_name
            target_subdir.mkdir(parents=True, exist_ok=True)

            if subdir_name in events_subdirs:
                events_subdir = events_subdirs[subdir_name]
                for file in events_subdir.iterdir():
                    if file.is_file():
                        target_file = target_subdir / file.name
                        try:
                            shutil.copy2(file, target_file)
                            stats['events_files_copied'] += 1
                        except Exception as e:
                            stats['errors'].append(f"Error copying {file}: {e}")

            artifacts_target_dir = target_subdir / "artifacts"
            artifacts_target_dir.mkdir(parents=True, exist_ok=True)

            if subdir_name in artifacts_subdirs:
                artifacts_subdir = artifacts_subdirs[subdir_name]
                for file in artifacts_subdir.iterdir():
                    if file.is_file():
                        target_file = artifacts_target_dir / file.name
                        try:
                            shutil.copy2(file, target_file)
                            stats['artifacts_files_copied'] += 1
                        except Exception as e:
                            stats['errors'].append(f"Error copying {file}: {e}")

            stats['subdirs_processed'] += 1

        stats['domains_processed'] += 1
        print(f"Completed {domain}: {len(all_subdirs)} subdirectories processed")

    print("
" + "=" * 80)
    print("ORGANIZATION SUMMARY")
    print("=" * 80)
    print(f"Domains processed: {stats['domains_processed']}")
    print(f"Subdirectories processed: {stats['subdirs_processed']}")
    print(f"Events files copied: {stats['events_files_copied']}")
    print(f"Artifacts files copied: {stats['artifacts_files_copied']}")
    print(f"Total files copied: {stats['events_files_copied'] + stats['artifacts_files_copied']}")

    if stats['errors']:
        print(f"
Errors encountered: {len(stats['errors'])}")
        for error in stats['errors'][:10]:
            print(f"  - {error}")
        if len(stats['errors']) > 10:
            print(f"  ... and {len(stats['errors']) - 10} more errors")
    else:
        print("
No errors encountered!")

    return stats

stats = organize_files()


In [ ]:
import shutil
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
if not (NOTEBOOK_DIR / "organize.ipynb").exists():
    NOTEBOOK_DIR = Path("data_pipeline/scenario_assembly").resolve()
PIPELINE_DIR = NOTEBOOK_DIR.parent

# Source directory for concepts
CONCEPTS_SRC = PIPELINE_DIR / "concept_generation" / "output"

# Target directory
TARGET_DIR = NOTEBOOK_DIR

# Domains to process
DOMAINS = ["research", "tutoring"]

def organize_concepts():
    """Organize concept files into the scenario assembly directory structure."""

    stats = {
        'domains_processed': 0,
        'subdirs_processed': 0,
        'concepts_files_copied': 0,
        'errors': []
    }

    for domain in DOMAINS:
        print("
" + "=" * 80)
        print(f"Processing {domain.upper()} domain - CONCEPTS")
        print("=" * 80)

        concepts_domain_path = CONCEPTS_SRC / domain
        target_domain_path = TARGET_DIR / domain

        if not concepts_domain_path.exists():
            print(f"Warning: {concepts_domain_path} does not exist, skipping...")
            continue
        if not target_domain_path.exists():
            print(f"Warning: {target_domain_path} does not exist, skipping...")
            continue

        concepts_subdirs = {d.name: d for d in concepts_domain_path.iterdir() if d.is_dir()}

        print(f"Found {len(concepts_subdirs)} concept subdirectories")
        print("Processing subdirectories...")

        for idx, subdir_name in enumerate(sorted(concepts_subdirs.keys(), key=lambda x: int(x) if x.isdigit() else float('inf')), 1):
            if idx % 20 == 0 or idx == len(concepts_subdirs):
                print(f"  Progress: {idx}/{len(concepts_subdirs)} subdirectories")

            target_subdir = target_domain_path / subdir_name
            if not target_subdir.exists():
                print(f"  Warning: Target subdirectory {target_subdir} does not exist, skipping...")
                continue

            concept_target_dir = target_subdir / "concept"
            concept_target_dir.mkdir(parents=True, exist_ok=True)

            concept_subdir = concepts_subdirs[subdir_name]
            for file in concept_subdir.iterdir():
                if file.is_file() and file.suffix == '.json':
                    target_file = concept_target_dir / file.name
                    try:
                        shutil.copy2(file, target_file)
                        stats['concepts_files_copied'] += 1
                    except Exception as e:
                        stats['errors'].append(f"Error copying {file}: {e}")

            stats['subdirs_processed'] += 1

        stats['domains_processed'] += 1
        print(f"Completed {domain}: {len(concepts_subdirs)} subdirectories processed")

    print("
" + "=" * 80)
    print("CONCEPT ORGANIZATION SUMMARY")
    print("=" * 80)
    print(f"Domains processed: {stats['domains_processed']}")
    print(f"Subdirectories processed: {stats['subdirs_processed']}")
    print(f"Concepts files copied: {stats['concepts_files_copied']}")

    if stats['errors']:
        print(f"
Errors encountered: {len(stats['errors'])}")
        for error in stats['errors'][:10]:
            print(f"  - {error}")
        if len(stats['errors']) > 10:
            print(f"  ... and {len(stats['errors']) - 10} more errors")
    else:
        print("
No errors encountered!")

    return stats

concept_stats = organize_concepts()
